In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)

In [28]:
# user_behaviour_features.ipynb - Lecture du fichier unique amazon_reviews_cleaned.csv

import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

file_path = Path("/Users/tsantaloic/Desktop/Projet_05_AIA02/data/processed/amazon_reviews_cleaned.csv")

# Vérifications
if not file_path.exists():
    raise FileNotFoundError(f"Fichier introuvable: {file_path}")
if not file_path.is_file():
    raise ValueError(f"Le chemin ne pointe pas vers un fichier: {file_path}")

# Lecture
df = pd.read_csv(file_path)

# Optionnel: ajouter une colonne catégorie basée sur le nom du fichier
#df["category"] = file_path.stem

# Infos rapides
print("Chemin:", file_path)
print("Shape:", df.shape)
display(df.head())


Chemin: /Users/tsantaloic/Desktop/Projet_05_AIA02/data/processed/amazon_reviews_cleaned.csv
Shape: (20990, 14)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category,user_id_enc,item_id_enc,rating_norm
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,"[{'attachment_type': 'IMAGE', 'large_image_url...",B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True,Electronics,1456,15167,-1.239889
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True,Electronics,1456,13689,-3.037525
2,5.0,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True,Electronics,1456,10383,0.557747
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,2010-11-20 18:41:35.000,18,True,Electronics,2248,5393,0.557747
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,2023-02-17 02:39:41.238,0,True,Electronics,1892,12523,0.557747


In [29]:
df.duplicated().sum()


np.int64(0)

In [30]:
df = df.drop_duplicates()


In [31]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20990 entries, 0 to 20989
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rating             20990 non-null  float64
 1   title              20990 non-null  object 
 2   text               20990 non-null  object 
 3   images             20990 non-null  object 
 4   asin               20990 non-null  object 
 5   parent_asin        20990 non-null  object 
 6   user_id            20990 non-null  object 
 7   timestamp          20990 non-null  object 
 8   helpful_vote       20990 non-null  int64  
 9   verified_purchase  20990 non-null  bool   
 10  category           20990 non-null  object 
 11  user_id_enc        20990 non-null  int64  
 12  item_id_enc        20990 non-null  int64  
 13  rating_norm        20990 non-null  float64
dtypes: bool(1), float64(2), int64(3), object(8)
memory usage: 2.1+ MB


In [32]:
df.isna().sum().sort_values(ascending=False)


rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
category             0
user_id_enc          0
item_id_enc          0
rating_norm          0
dtype: int64

In [33]:
# Nombres d'achats & recurrence par utilisateur
user_stats = df.groupby("user_id").agg(
    nb_interactions=("parent_asin", "count"),
    avg_rating=("rating", "mean"),
    rating_std=("rating", "std"),
    last_interaction=("timestamp", "max")
).reset_index()
user_stats["rating_std"] = user_stats["rating_std"].fillna(0.0)


user_stats.head()


,user_id,nb_interactions,avg_rating,rating_std,last_interaction
0,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,1,5.000000,0.000000,2018-03-03 01:11:55.541
1,AE23YCERVBZ7ABKTESKSSXYTLYRA,1,5.000000,0.000000,2009-12-15 14:48:27.000
2,AE246WPAV4HBLUOOOKTECTYQ7ZPA,2,5.000000,0.000000,2023-03-16 11:09:16.918
3,AE25NQAZI3725GZIL5FS52ZIKWKQ,9,4.555556,1.333333,2019-11-14 17:30:08.565
4,AE26ICWKMFJEHDH5VDX4W42H2NMA,3,4.333333,0.577350,2021-07-29 13:24:38.824


In [39]:
# Préférences de catégories 
user_category_dist = (
    df.groupby(["user_id", "category"])
      .size()
      .unstack(fill_value=0)
)

user_category_dist.head()

# chiffre = nbr d'interactions avrc la categorie

category,Automotive,Books,CDs_and_Vinyl,Cell_Phones_and_Accessories,Clothing_Shoes_and_Jewelry,Digital_Music,Electronics
user_id,,,,,,,
AE225Z2VRWT6GPTOMA4H4O3H2KVQ,0,0,0,0,0,1,0
AE23YCERVBZ7ABKTESKSSXYTLYRA,0,0,0,0,0,1,0
AE246WPAV4HBLUOOOKTECTYQ7ZPA,0,0,0,2,0,0,0
AE25NQAZI3725GZIL5FS52ZIKWKQ,9,0,0,0,0,0,0
AE26ICWKMFJEHDH5VDX4W42H2NMA,0,0,0,0,0,0,3
